# Real-data validation: Paganin reconstruction on I12 dataset 6

Apply Paganin phase retrieval to the biological sample in `i12_dataset6.npz`
and reconstruct five central slices with FBP3D. Compare the sinograms,
reconstructed slices and profiles with and without phase retrieval.

## Data and execution

Use `i12_dataset6.npz` from [Zenodo record 17252190](https://zenodo.org/records/17252190).
The repository downloader verifies the archive checksums.
Run from the repository root:

```bash
python .scripts/download_zenodo.py Zenodo
python -m pip install -e '.[validation]'
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/preprocessing/paganin_reconstruction.ipynb
```

Use an NVIDIA GPU with compatible CuPy and ToMoBAR installed, plus ASTRA for
ASTRA-based reconstruction. Run notebooks serially. Set `HTTOMOLIBGPU_DATA_DIR`
to use an archive stored elsewhere.

Documentation builds render saved outputs without running GPU computations.
Execute with `--overwrite` before building documentation to update the results.


In [ ]:
import gc
import hashlib
import json
import os
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log
from httomolibgpu.prep.phase import paganin_filter
from httomolibgpu.recon.algorithm import FBP3d_tomobar
from httomolibgpu.recon.rotation import find_center_vo

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")
data_path = data_dir / "i12_dataset6.npz"
if not data_path.is_file():
    raise FileNotFoundError(
        f"Missing {data_path}; run .scripts/download_zenodo.py first"
    )
with data_path.open("rb") as stream:
    checksum = hashlib.file_digest(stream, "sha256").hexdigest()
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print(
    json.dumps(
        {
            "dataset": data_path.name,
            "sha256": checksum,
            "cupy": cp.__version__,
            "numpy": np.__version__,
            "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
            "gpu": (
                props["name"].decode()
                if isinstance(props["name"], bytes)
                else props["name"]
            ),
        },
        indent=2,
    )
)

with np.load(data_path, allow_pickle=False) as archive:
    projections = archive["projdata"]
    flats = archive["flats"]
    darks = archive["darks"]
    angles = np.deg2rad(archive["angles"]).astype(np.float32)

n_angles, n_rows, n_columns = projections.shape
assert n_rows >= 5 and angles.shape == (n_angles,)
assert flats.shape[1:] == darks.shape[1:] == (n_rows, n_columns)
row_start = n_rows // 2 - 2
selected_rows = slice(row_start, row_start + 5)
print("Projection shape:", projections.shape)
print("Five reconstructed detector rows:", list(range(row_start, row_start + 5)))

## Phase retrieval before reconstruction

The archive stores projections as `(angle, detector row, detector column)` and
angles in degrees. Correct the transmission using flats and darks first.
`paganin_filter` includes the negative logarithm: **do not apply `minus_log` to
its output**.

Use the configured example settings: 30 µm pixel size, 3.2 m propagation
distance, 53 keV beam energy and `ratio_delta_beta=500`. The archive does not
contain acquisition metadata; confirm pixel size and distance before drawing
quantitative physical conclusions. The ratio is an illustrative single-material
assumption, not a measured material constant.
Adjust it for the sample when interpreting the smoothing/resolution tradeoff.

Process batches of 32 projections, retaining all 50 available detector rows
while filtering, then keep just the five central rows for reconstruction.
This avoids introducing a five-row crop boundary before the two-dimensional
phase filter; the finite vertical extent of the archive still limits context.
No stripe-removal step is applied, so the comparison isolates phase retrieval.

In [ ]:
phase_parameters = {
    "pixel_size": 30,  # micrometres
    "distance": 3.2,  # metres
    "energy": 53.0,  # keV
    "ratio_delta_beta": 500.0,
    "calculate_padding_value_method": "next_power_of_2",
}
print("Paganin parameters:", phase_parameters)
batch_size = 32
absorption = cp.empty((n_angles, 5, n_columns), dtype=cp.float32)
phase_retrieved = cp.empty_like(absorption)
flats_gpu = cp.asarray(flats)
darks_gpu = cp.asarray(darks)
for start in range(0, n_angles, batch_size):
    stop = min(start + batch_size, n_angles)
    transmission = dark_flat_field_correction(
        cp.asarray(projections[start:stop]), flats_gpu, darks_gpu
    )
    absorption[start:stop] = minus_log(transmission[:, selected_rows, :].copy())
    filtered = paganin_filter(transmission, **phase_parameters)
    phase_retrieved[start:stop] = filtered[:, selected_rows, :]
    del transmission, filtered

# del projections, flats, darks, flats_gpu, darks_gpu
cp.get_default_memory_pool().free_all_blocks()
for data in (absorption, phase_retrieved):
    assert data.shape == (n_angles, 5, n_columns)
    assert data.dtype == cp.float32
    assert bool(cp.isfinite(data).all()), "Non-finite preprocessed values"

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, data, title in zip(
    axes,
    (absorption, phase_retrieved),
    ("Without phase retrieval", "With Paganin phase retrieval"),
):
    image = cp.asnumpy(data[:, 2, :])
    lo, hi = np.percentile(image, [1, 99])
    im = ax.imshow(image, cmap="gray", aspect="auto", vmin=lo, vmax=hi)
    ax.set(title=title, xlabel="Detector column", ylabel="Projection index")
    fig.colorbar(im, ax=ax, label="Negative log transmission")
plt.show()
plt.close(fig)

## Five-slice FBP reconstruction

Estimate one rotation centre from the central absorption sinogram and use it
for both reconstructions. Use 750 pixels of detector padding and a SINC cutoff
of 1.0. Output axes are
`(Y, detector row, X)`. Reconstruct the two cases sequentially to limit GPU memory.
These checks verify shape and finite values; image quality requires visual
review and is not established by these checks alone.

In [ ]:
center = float(find_center_vo(absorption[:, 2, :], smin=-150, smax=150))
assert np.isfinite(center) and 0 < center < n_columns
reconstruction_parameters = {
    "center": center,
    "detector_pad": 750,
    "filter_freq_cutoff": 1.0,
    "recon_mask_radius": 2.0,
}
print("Reconstruction parameters:", reconstruction_parameters)
reconstructions = []
for data in (absorption, phase_retrieved):
    reconstruction = FBP3d_tomobar(data, angles, **reconstruction_parameters)
    assert reconstruction.shape == (n_columns, 5, n_columns)
    assert reconstruction.dtype == cp.float32
    assert reconstruction.flags.c_contiguous
    assert bool(cp.isfinite(reconstruction).all()), "Non-finite reconstruction"
    reconstructions.append(cp.asnumpy(reconstruction))
    del reconstruction
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
del absorption, phase_retrieved, data
cp.get_default_memory_pool().free_all_blocks()
print("Both five-slice reconstructions passed structural and finite-value checks.")

## Reconstructed slices and line profile

Show all five slices, with a shared display range across both methods, then
compare the central horizontal profiles. Paganin retrieval typically suppresses
phase fringes and noise while smoothing fine structures; its single-material
assumption can bias boundaries in heterogeneous biological specimens.
Values are in library reconstruction units, without absolute attenuation calibration.

In [ ]:
lo, hi = np.percentile(reconstructions[0][:, 2, :], [1, 99])
fig, axes = plt.subplots(2, 5, figsize=(18, 8), constrained_layout=True)
for row, (volume, label) in enumerate(
    zip(reconstructions, ("Without phase retrieval", "With Paganin"))
):
    for i in range(5):
        im = axes[row, i].imshow(volume[:, i, :], cmap="gray", vmin=0.9 * lo, vmax=hi)
        axes[row, i].set(title=f"{label}: row {row_start + i}", xlabel="X pixel")
        axes[row, i].set_ylabel("Y pixel")
fig.colorbar(im, ax=axes.ravel().tolist(), label="Reconstructed value (library units)")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 3), constrained_layout=True)
for volume, label in zip(reconstructions, ("Without phase retrieval", "With Paganin")):
    ax.plot(volume[volume.shape[0] // 2, 2, :], label=label, linewidth=1)
ax.set(
    title="Central slice: horizontal profile",
    xlabel="X pixel",
    ylabel="Reconstructed value (library units)",
)
ax.legend()
ax.grid(alpha=0.3)
plt.show()
plt.close(fig)
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

## Final validation

Check both five-slice reconstructions after plotting: shape, dtype,
finite and non-constant values, and a measurable change after phase retrieval.
These checks detect invalid or unchanged outputs; they do not validate physical
calibration or image quality. No fixed numerical regression baseline is available.

In [ ]:
assert len(reconstructions) == 2, "Both comparison reconstructions are required"
for label, volume in zip(("Without phase retrieval", "With Paganin"), reconstructions):
    assert volume.shape == (n_columns, 5, n_columns), f"Unexpected shape: {label}"
    assert volume.dtype == np.float32, f"Unexpected dtype: {label}"
    assert volume.flags.c_contiguous, f"Non-contiguous output: {label}"
    assert np.isfinite(volume).all(), f"Non-finite reconstruction: {label}"
    assert np.max(volume) > np.min(volume), f"Constant reconstruction: {label}"
    print(
        label,
        {
            "mean": float(np.mean(volume, dtype=np.float64)),
            "minimum": float(np.min(volume)),
            "maximum": float(np.max(volume)),
        },
    )
assert not np.allclose(
    reconstructions[0], reconstructions[1], rtol=1e-5, atol=1e-8
), "Phase retrieval produced no measurable change in the reconstruction"
print("Both five-slice reconstruction validation checks passed.")
del reconstructions
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()